# Fine-Tuning TinyLlama for Sentiment Classification

This chapter documents the fine-tuning setup for TinyLlama on the sentiment task. Code execution is disabled in Quarto because training will be run in Colab, not during book rendering.

## Install dependencies

This block installs the libraries needed for QLoRA fine-tuning in Colab. We use `transformers` for the base model, `peft` for LoRA adapters, `trl` for supervised fine-tuning, and `bitsandbytes` for 4-bit quantization.

In [1]:
#| label: install-dependencies
#| echo: true

!pip install -q transformers peft trl bitsandbytes accelerate datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00


## Import packages

This block imports the core libraries for loading data, configuring the model, fine-tuning with QLoRA, and evaluating the predictions.

In [2]:
#| label: imports
#| echo: true

import pandas as pd
import numpy as np
import torch

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig
from trl import SFTTrainer

from sklearn.metrics import f1_score, classification_report

## Local paths for Positron / Quarto

This block resolves the repository root and constructs local file paths. It is useful when writing and documenting the notebook in Positron before converting it to a Colab notebook.

In [3]:
#| label: paths-and-config
#| echo: true

# _cwd = Path.cwd()
# REPO_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parents[1]
# STAGE2_DIR = REPO_ROOT / "data/processed/stage2"

SEED = 42

# train_path = STAGE2_DIR / "llm_train.parquet"
# test_path = STAGE2_DIR / "llm_test.parquet"

For usage in **Positron/Quarto**, use the next block. For **Google Colab**, use the Colab block further below instead.

## Load data locally

This block loads the train and test files from the local repository structure. It is only used in the local Quarto / Positron workflow.

In [ ]:
#| label: load-data
#| echo: true

# train_df = pd.read_parquet(train_path)
# test_df = pd.read_parquet(test_path)

# train_df.shape, test_df.shape

## Alternative: load data in Colab

This alternative block mounts Google Drive and loads the train and test parquet files from the shared Colab folder. Use this block in Colab instead of the local path/loading blocks above.

In [29]:
#| label: colab-drive-and-load-data
#| echo: true

from google.colab import drive

drive.mount('/content/drive')

train_path = "/content/drive/MyDrive/CLT/stage2/llm_train.parquet"
test_path = "/content/drive/MyDrive/CLT/stage2/llm_test.parquet"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

train_df.shape, test_df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


((13053, 6), (3264, 6))

In [ ]:
train_df["sentiment_label"] = train_df["sentiment_label"].str.replace("(partly) negative", "negative")
test_df["sentiment_label"] = test_df["sentiment_label"].str.replace("(partly) negative", "negative")

## Inspect the data

This block shows a few examples of the prompt text and the target sentiment label. It verifies that the dataset is already in instruction format and ready for supervised fine-tuning.

In [30]:
#| label: inspect-data
#| echo: true

train_df[["text", "sentiment_label"]].head(3)

,text,sentiment_label
0,### Instruction:\nYou are an AI media analyst....,neutral
1,### Instruction:\nYou are an AI media analyst....,neutral
2,### Instruction:\nYou are an AI media analyst....,neutral


## Define labels and model name

This block defines the label set, the input and target columns, and the selected base model. Keeping these settings in one place makes the notebook easier to adjust later.

In [31]:
#| label: define-labels
#| echo: true

LABELS = ["negative", "neutral", "positive"]
TARGET_COLUMN = "sentiment_label"
TEXT_COLUMN = "text"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

## Configure 4-bit quantization

This block sets up the quantization configuration for QLoRA. We use 4-bit loading to reduce GPU memory usage, which is important for running TinyLlama efficiently on Colab Pro.

In [33]:
#| label: quantization-config
#| echo: true

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

## Load tokenizer and model

This block loads the tokenizer and the quantized base model. If the tokenizer has no padding token, we reuse the EOS token so batching works correctly.

In [34]:
#| label: load-tokenizer-and-model
#| echo: true

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Configure LoRA adapters

This block defines the LoRA setup. The chosen values are conservative starter settings: small enough for a first Colab test, but still standard for parameter-efficient fine-tuning.

In [35]:
#| label: lora-config
#| echo: true

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## Prepare dataset

This block prepares the full training and test datasets for fine-tuning and evaluation. The `text` column contains the Alpaca-formatted instruction, the `sentiment_label` column contains the gold label used for evaluation after inference.

In [36]:
#| label: prepare-datasets
#| echo: true

train_dataset = Dataset.from_pandas(train_df[[TEXT_COLUMN]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[[TEXT_COLUMN, TARGET_COLUMN]], preserve_index=False)

## Define training arguments

This block sets small training parameters for the initial test run. The batch size is intentionally small to fit Colab GPU memory, and we train only for one epoch to validate the pipeline quickly.

In [37]:
#| label: training-args
#| echo: true

training_args = TrainingArguments(
    output_dir="outputs/tinyllama-sentiment",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    bf16=True,
    fp16=False,
)

## Create the supervised fine-tuning trainer

This block combines the base model, LoRA configuration, training dataset, and training arguments into one trainer object. The trainer uses the `text` column as the input field.

In [38]:
#| label: sft-trainer
#| echo: true

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/13053 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13053 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2187 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/13053 [00:00<?, ? examples/s]

## Run the first training pass

This block starts fine-tuning. In the first Colab test, it should only confirm that the pipeline runs successfully on the small debug sample.

In [39]:
#| label: run-training
#| echo: true

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.361218
20,2.260621
30,2.149374
40,2.170698
50,2.173380
60,2.117432
70,2.154802
80,2.191675
90,2.137134
100,2.098965


TrainOutput(global_step=4896, training_loss=2.046169179132561, metrics={'train_runtime': 3944.9077, 'train_samples_per_second': 9.926, 'train_steps_per_second': 1.241, 'total_flos': 2.4943680782512128e+17, 'train_loss': 2.046169179132561})

## Generate predictions on the debug test set

This block applies the fine-tuned model to a few held-out examples. We use only a small number of prompts and keep the generation short because the target output is just one sentiment label.

In [43]:
#| label: make-inference-prompt
#| echo: true

def make_inference_prompt(full_text: str) -> str:
    parts = full_text.split("### Response:")
    instruction = parts[0]
    words = instruction.split()
    if len(words) > 500:
        instruction = " ".join(words[:500]) + "\n\n"
    return instruction + "### Response:"

In [44]:
#| label: generate-predictions
#| echo: true

generator = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=12,
    return_full_text=False,
)

sample_prompts = [make_inference_prompt(t) for t in test_df[TEXT_COLUMN].tolist()]
sample_outputs = generator(sample_prompts)

sample_outputs

Both `max_new_tokens` (=12) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[[{'generated_text': ' Uncertainty is a fundamental aspect of AI. When'}],
 [{'generated_text': '\nBased on the given text material, the text material'}],
 [{'generated_text': ' Negative\n\n##### Assistant\nThe most important'}],
 [{'generated_text': ' Negative'}],
 [{'generated_text': ' Negative'}],
 [{'generated_text': ' negative\n'}],
 [{'generated_text': ' Positive'}],
 [{'generated_text': ' Negative\n( partly)\n\n### Input:'}],
 [{'generated_text': ' NegativeClaim: Yes, I am aware of these'}],
 [{'generated_text': ' Positive", \'It is an AI-powered'}],
 [{'generated_text': ' Positive'}],
 [{'generated_text': ' Negative'}],
 [{'generated_text': ' Error: invalid payload, expected ` object`, got ` array'}],
 [{'generated_text': ' Unresponsive or Undetectable COVID-19'}],
 [{'generated_text': ' 0 negative, 0 neutral, 10 positive'}],
 [{'generated_text': ' positive'}],
 [{'generated_text': '\nneutral'}],
 [{'generated_text': ' Unicorns are getting big. The latest big un'}],
 [{'generat

## Extract labels from model outputs

This block maps the generated text back to one of the three sentiment labels. Because we set `return_full_text=False`, the extraction runs only on the model output, not on the original prompt.

In [48]:
#| label: extract-labels
#| echo: true

def extract_label(generated_text: str) -> str:
    text = generated_text.strip().lower()
    if "negative" in text:
        return "negative"
    elif "neutral" in text:
        return "neutral"
    elif "positive" in text:
        return "positive"
    else:
        return "unknown"

pred_labels = []
true_labels = test_df[TARGET_COLUMN].tolist()

for output in sample_outputs:
    generated_text = output[0]["generated_text"]
    pred_labels.append(extract_label(generated_text))

list(zip(true_labels, pred_labels))

[('neutral', 'unknown'),
 ('neutral', 'unknown'),
 ('(partly) negative', 'unknown'),
 ('positive', '(partly) negative'),
 ('positive', '(partly) negative'),
 ('neutral', '(partly) negative'),
 ('positive', 'positive'),
 ('(partly) negative', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', 'positive'),
 ('positive', 'positive'),
 ('positive', '(partly) negative'),
 ('positive', 'unknown'),
 ('neutral', 'unknown'),
 ('positive', 'neutral'),
 ('positive', 'positive'),
 ('neutral', 'neutral'),
 ('positive', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', 'neutral'),
 ('neutral', 'unknown'),
 ('neutral', 'positive'),
 ('neutral', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', 'positive'),
 ('neutral', 'unknown'),
 ('positive', 'unknown'),
 ('neutral', 'positive'),
 ('neutral', 'unknown'),
 ('neutral', 'positive'),
 ('positive', 'unknown'),
 ('neutral', 'unknown'),
 ('neutral', '(partly) negative'),
 ('positive', '(partly) negative'),


## Compute Macro-F1

This block computes Macro-F1 on the predictions that could be mapped to one of the valid labels. Macro-F1 is the main evaluation metric because it gives equal weight to all sentiment classes.

In [49]:
#| label: evaluate-macro-f1
#| echo: true

valid_idx = [i for i, pred in enumerate(pred_labels) if pred in LABELS]

y_true = [true_labels[i] for i in valid_idx]
y_pred = [pred_labels[i] for i in valid_idx]

macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro")

print(f"Valid predictions: {len(valid_idx)} / {len(pred_labels)}")
print("Macro-F1:", macro_f1)
print(classification_report(y_true, y_pred, labels=LABELS))

Valid predictions: 1783 / 3264
Macro-F1: 0.3022393995254846
                   precision    recall  f1-score   support

(partly) negative       0.15      0.55      0.24       235
          neutral       0.61      0.19      0.29      1052
         positive       0.34      0.42      0.38       496

         accuracy                           0.30      1783
        macro avg       0.37      0.39      0.30      1783
     weighted avg       0.48      0.30      0.31      1783



## Notes

For the first Colab test, this notebook should be run on the small debug sample. After the pipeline works end to end, the same setup can be scaled to the full training and test data.